# Stabilitätsüberprüfung

## Zweck

Dieses Notebook überprüft die zeitliche Stabilität der Modellkoeffizienten
zwischen dem Campus File (2010) und den Daten des InEK-Datenbrowsers (2024).
Es beantwortet damit die Frage, ob die auf 2010er Daten trainierten Modelle
auch für aktuellere Daten valide sind.

Das Notebook gliedert sich in drei Abschnitte:

1. **Gesamtstabilitätsbericht** über alle ICD-Gruppen und -Kapitel
2. **Identifikation der einflussreichsten ICD-Merkmale** für die Baseline-Konfiguration
3. **Zielgerichtete Stabilitätsprüfung** für die einflussreichsten Gruppen und Kapitel

## Voraussetzung

Das Auswertungs-Notebook muss vollständig ausgeführt worden sein
(HTML-Berichte der Baseline-Konfiguration müssen vorliegen).

In [1]:
from drg_risk.analysis.stability_analyzer import StabilityAnalyzer
from drg_risk import REPORT_DIR

import os
import pandas as pd

E:\Uni\Master\007Masterarbeit\drg_risk\src\drg_risk\analysis\stability_analyzer.py:567: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  
E:\Uni\Master\007Masterarbeit\drg_risk\src\drg_risk\analysis\stability_analyzer.py:566: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  """.replace("icd_group", "ICD-Gruppe").replace("icd_chapter", "ICD-Kapitel").replace("[],", "")


## 1. Gesamtstabilitätsbericht

Erstellt einen Stabilitätsbericht über alle ICD-Gruppen und -Kapitel.

In [2]:
stab_ana = StabilityAnalyzer()

stab_ana.check_stability()

2026-09-30 21:58:09,803 - Daten werden geladen.
2026-09-30 21:58:09,807 - Initialisierung des Processors abgeschlossen.
2026-09-30 21:58:09,808 - Lade Feather Daten
2026-09-30 21:58:10,104 - Lade bereinigte ICD-Gruppen.
2026-09-30 21:58:10,122 - Bereinigte ICD-Gruppen geladen.
2026-09-30 21:58:10,122 - Lade bereinigte OPS-Gruppen.
2026-09-30 21:58:10,123 - Bereinigte OPS-Gruppen geladen.
2026-09-30 21:58:10,124 - Lade bereinigte DRG-Codes.
2026-09-30 21:58:10,348 - Bereinigte DRG-Codes geladen.
2026-09-30 21:58:10,349 - Lade Metadaten der Spalten.
2026-09-30 21:58:10,352 - Metadaten geladen.
2026-09-30 21:58:10,353 - Ungruppiertes Campus File geladen
2026-09-30 21:58:10,354 - Campus File geladen.
2026-09-30 21:58:10,355 - Verarbeite Campus File weiter.
2026-09-30 21:58:25,917 - Campus File weiterverarbeitet.
2026-09-30 21:58:25,918 - Lade Datenbrowser Ergebnisse.
2026-09-30 21:59:26,948 - Datenbrowser Ergebnisse geladen.
2026-09-30 21:59:26,956 - Datenquellen zusammengeführt.
2026-09-3

E:\Uni\Master\007Masterarbeit\drg_risk\src\drg_risk\analysis\stability_analyzer.py:836: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  res = stats.spearmanr(df_combined["campus_file_2010"], df_combined["datenbrowser_2024"])[0]


## 2. Einflussreichste ICD-Merkmale der Baseline

Liest die Regressionskoeffizienten der Baseline-Konfiguration aus den
gespeicherten HTML-Berichten und gibt jeweils die drei ICD-Merkmale mit
dem größten absoluten Koeffizient aus — getrennt nach Zielgröße
(Mortalität, Verweildauer) und ICD-Ebene (Gruppe, Kapitel).

Die so identifizierten Merkmale werden im dritten Abschnitt gezielt
auf Stabilität geprüft.

In [3]:
# Hyperparameter der Baseline
output_vars = ["entl_grd", "typ_vwd"]
subset = "gesamt"
input_var = "codes"
sdx = True
kovarianten = True
icd_structures = ["group", "chapter"]

# Herausfinden, welche Kapitel und Gruppen den größten Einfluss haben
for output_var in output_vars:
    for icd_structure in icd_structures:
        path = os.path.join(REPORT_DIR, "scores_postprocessed", f"output_var={output_var}", f"subset={subset}", f"input_var={input_var}", f"sdx={sdx}", f"{icd_structure}_kovarianten={kovarianten}.html")

        if output_var == "entl_grd":
            print("=== Mortalität ===")
        else:
            print("=== Verweildauer")

        if icd_structure == "group":
            print("--- Gruppen ---")
        else:
            print("--- Kapitel ---")

        tabellen = pd.read_html(path)

        df = tabellen[-1]

        df = df.drop(["p_value", "Unnamed: 0"], axis = 1)
        df = df.rename(columns = {"name": "Name", "coefficient": "Koeffizient", "odds_ratio": "OR", "std_error": "Standardfehler"})

        df = df[[name.startswith("icd") for name in df["Name"]]]

        df_sorted = df.sort_values(by="Koeffizient", key=abs, ascending=False)

        print(df_sorted["Name"].str.replace("icd_group_is_", "Gruppe: ", regex=True).str.replace("icd_chapter_is_", "Kapitel: ").head(3).to_string(index=False))


=== Mortalität ===
--- Gruppen ---
Gruppe: I10-I15
Gruppe: J95-J99
Gruppe: M40-M54
=== Mortalität ===
--- Kapitel ---
Kapitel: 10
Kapitel: 16
 Kapitel: 9
=== Verweildauer
--- Gruppen ---
Gruppe: D60-D64
Gruppe: B95-B97
Gruppe: M00-M25
=== Verweildauer
--- Kapitel ---
 Kapitel: 3
 Kapitel: 9
Kapitel: 18


## 3. Zielgerichtete Stabilitätsprüfung

Stabilitätsprüfung für die einflussreichsten ICD-Gruppen und -Kapitel,
identifiziert in Abschnitt 2 — getrennt nach Zielgröße.

In [4]:
# Mortalität
stab_ana.check_stability(icd_groups=["I10-I15"])
stab_ana.check_stability(icd_groups=["J95-J99"])
stab_ana.check_stability(icd_groups=["M40-M54"])
stab_ana.check_stability(icd_chapters=["10"])
stab_ana.check_stability(icd_chapters=["16"])
stab_ana.check_stability(icd_chapters=["9"])

# Verweildauer
stab_ana.check_stability(icd_groups=["D60-D64"])
# stab_ana.check_stability(icd_groups=["B95-B97"]), kein Vergleich möglich, da die Fallzahl 2024 zu gering war und keine Daten vorliegen
stab_ana.check_stability(icd_groups=["M00-M25"])
stab_ana.check_stability(icd_chapters=["3"])
stab_ana.check_stability(icd_chapters=["9"])
stab_ana.check_stability(icd_chapters=["18"])

2026-09-30 21:59:29,765 - Starte Stabilitätsanalyse.


2026-09-30 21:59:29,767 - Berechne prozentuale Veränderungen.
2026-09-30 21:59:29,781 - Berechne Änderung der Geschlechterverteilung.
2026-09-30 21:59:29,788 - Berechne Änderung der Verweildauer.
2026-09-30 21:59:29,802 - Berechne Änderung der Altersstruktur.
2026-09-30 21:59:29,812 - Berechne Änderungen im Kodierverhalten.
2026-09-30 21:59:29,828 - Erstelle Report.
2026-09-30 21:59:29,884 - Report erstellt.
2026-09-30 21:59:29,884 - Stabilitätsanalyse abgeschlossen.
2026-09-30 21:59:29,885 - Starte Stabilitätsanalyse.
2026-09-30 21:59:29,887 - Berechne prozentuale Veränderungen.
2026-09-30 21:59:29,902 - Berechne Änderung der Geschlechterverteilung.
2026-09-30 21:59:29,909 - Berechne Änderung der Verweildauer.
2026-09-30 21:59:29,923 - Berechne Änderung der Altersstruktur.
2026-09-30 21:59:29,934 - Berechne Änderungen im Kodierverhalten.
2026-09-30 21:59:29,950 - Erstelle Report.
2026-09-30 21:59:30,006 - Report erstellt.
2026-09-30 21:59:30,007 - Stabilitätsanalyse abgeschlossen.
202